# RVC Training — Standalone Colab Notebook

Notebook huấn luyện giọng nói RVC độc lập trên Google Colab (Python 3.12).

**Chạy từng cell theo thứ tự từ trên xuống.**

| Cell | Bước |
|------|------|
| 1 | Mount Drive + Clone repo + Cài thư viện |
| 2 | ⚙️ Cấu hình (chỉnh sửa ở đây) |
| 3 | Symlinks → Drive + Tải weights |
| 4 | 🎙️ Chuẩn bị audio |
| 5 | 🔧 Tiền xử lý |
| 6 | 🎵 Trích F0 + HuBERT features |
| 7 | 🚀 Training |
| 8 | 📦 FAISS index + Export model |

In [2]:
# ── Cell 1: Mount Drive + Clone repo + Cài thư viện ─────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os, re, subprocess, sys
from pathlib import Path

DRIVE_DIR = '/content/drive/MyDrive/rvc_standalone'
REPO      = '/content/compare_rvc'

os.makedirs(DRIVE_DIR, exist_ok=True)

if os.path.isdir(os.path.join(REPO, '.git')):
    subprocess.run(['git', '-C', REPO, 'pull'], check=True)
else:
    subprocess.run(
        ['git', 'clone', 'https://github.com/thanhNgan13/compare_rvc', REPO],
        check=True
    )

os.chdir(REPO)
if REPO not in sys.path:
    sys.path.insert(0, REPO)
print('CWD:', os.getcwd())

# --- Cài thư viện từ requirements.txt ---
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', 'pip>=23.2,<24.1'],
    check=True
)

# Package cần override vì version trong requirements.txt không có wheel cho Python 3.12
# None = bỏ qua (auto-resolved theo package khác)
OVERRIDES = {
    'numba':    'numba>=0.59.0',   # 0.56.4 chỉ hỗ trợ Python ≤3.10
    'llvmlite': None,              # auto-cài theo numba
    'librosa':  'librosa>=0.10.0', # 0.9.1 phụ thuộc numba cũ
    'numpy':    'numpy>=1.24.0',   # 1.23.5 không có wheel cho Python 3.12
    'faiss_cpu': 'faiss-cpu',      # 1.7.3 không có wheel cho Python 3.12
}
SKIP = {
    'aria2', 'gradio', 'gradio_client',
    'onnxruntime', 'onnxruntime_gpu',
    'fastapi', 'uvicorn',
}

req_lines = (Path(REPO) / 'requirements.txt').read_text().splitlines()
pkgs = []
for line in req_lines:
    line = line.strip()
    if not line or line.startswith('#'):
        continue
    name = re.split(r'[>=<!;\s]', line)[0].lower().replace('-', '_')
    if name in SKIP:
        continue
    if name in OVERRIDES:
        replacement = OVERRIDES[name]
        if replacement:
            pkgs.append(replacement)
        # None → bỏ qua
    else:
        pkgs.append(line)

print('Installing', len(pkgs), 'packages...')

failed = []
for pkg in pkgs:
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', pkg],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    if r.returncode != 0:
        print('[FAIL]', pkg)
        print(r.stdout[-800:])
        failed.append(pkg)
    else:
        print('[OK]', pkg)

if failed:
    print()
    print('=== CAC PACKAGE LOI ===')
    for p in failed:
        print(' ', p)
    raise RuntimeError(str(len(failed)) + ' package(s) failed — xem chi tiet o tren')

print()
print('Cai dat xong!')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CWD: /content/compare_rvc
Installing 42 packages...
[OK] joblib>=1.1.0
[OK] numba>=0.59.0
[OK] numpy>=1.24.0
[OK] scipy
[OK] librosa>=0.10.0
[OK] fairseq==0.12.2
[OK] faiss-cpu
[OK] Cython
[OK] pydub>=0.25.1
[OK] soundfile>=0.12.1
[OK] ffmpeg-python>=0.2.0
[OK] tensorboardX
[OK] Jinja2>=3.1.2
[OK] json5
[OK] Markdown
[OK] matplotlib>=3.7.0
[OK] matplotlib-inline>=0.1.3
[OK] praat-parselmouth>=0.4.2
[OK] Pillow>=9.1.1
[OK] resampy>=0.4.2
[OK] scikit-learn
[OK] tensorboard
[OK] tqdm>=4.63.1
[OK] tornado>=6.1
[OK] Werkzeug>=2.2.3
[OK] uc-micro-py>=1.0.1
[OK] sympy>=1.11.1
[OK] tabulate>=0.8.10
[OK] PyYAML>=6.0
[OK] pyasn1>=0.4.8
[OK] pyasn1-modules>=0.2.8
[OK] fsspec>=2022.11.0
[OK] absl-py>=1.2.0
[OK] audioread
[OK] colorama>=0.4.5
[OK] pyworld==0.3.2
[OK] httpx
[OK] torchcrepe==0.0.20
[OK] torchfcpe
[OK] ffmpy==0.3.1
[OK] python-dotenv>=1.0.0
[OK] av

Cai dat 

In [3]:
# ── Cell 2: ⚙️ CẤU HÌNH — chỉnh sửa các giá trị ở đây ─────────────────────

# Tên experiment — sẽ tạo thư mục logs/<EXPERIMENT> để chứa log và checkpoint
EXPERIMENT   = 'my_voice'

# Nguồn audio:
#   'upload' → tải file trực tiếp qua hộp upload (xuất hiện ở Cell 4)
#   'drive'  → lấy từ thư mục Drive trong DRIVE_AUDIO bên dưới
AUDIO_SRC    = 'drive'

# Đường dẫn thư mục .wav trên Drive (chỉ cần khi AUDIO_SRC='drive')
DRIVE_AUDIO  = DRIVE_DIR + '/audio/' + EXPERIMENT

# ── Tham số training ──────────────────────────────────────────────────────────
SAMPLE_RATE  = '40k'    # '32k' | '40k' | '48k'
VERSION      = 'v2'     # 'v1'  | 'v2'
F0_METHOD    = 'rmvpe'  # 'rmvpe' | 'harvest' | 'pm'
TOTAL_EPOCHS = 200
BATCH_SIZE   = 4
SAVE_EVERY   = 10       # lưu checkpoint mỗi N epoch

print('Config:')
print('  experiment  :', EXPERIMENT)
print('  sample_rate :', SAMPLE_RATE)
print('  version     :', VERSION)
print('  f0_method   :', F0_METHOD)
print('  total_epochs:', TOTAL_EPOCHS, ' batch_size:', BATCH_SIZE)

Config:
  experiment  : my_voice
  sample_rate : 40k
  version     : v2
  f0_method   : rmvpe
  total_epochs: 200  batch_size: 4


In [4]:
# ── Cell 3: Symlinks → Drive + Tải weights ───────────────────────────────────
import shutil, urllib.request
from pathlib import Path

RVC_ROOT   = Path('/content/compare_rvc')
DRIVE_ROOT = Path(DRIVE_DIR)

# --- Tạo symlinks assets / logs / datasets → Drive ---
print('=== Symlinks ===')
for name in ('assets', 'logs', 'datasets'):
    drive_path = DRIVE_ROOT / name
    drive_path.mkdir(parents=True, exist_ok=True)
    local = RVC_ROOT / name
    if local.is_symlink():
        local.unlink()
    elif local.is_dir():
        for item in local.iterdir():
            dst = drive_path / item.name
            if not dst.exists():
                shutil.move(str(item), str(dst))
        shutil.rmtree(str(local))
    local.symlink_to(drive_path)
    print(' ', name, '->', str(drive_path))

# --- Tải weights ---
print()
print('=== Tải weights ===')

def dl(url, dst):
    p = Path(dst)
    if p.exists():
        print('  skip (da co):', p.name)
        return
    p.parent.mkdir(parents=True, exist_ok=True)
    print('  downloading:', p.name)
    urllib.request.urlretrieve(url, p)
    print('  done:', p.name)

BASE = 'https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main'

dl(BASE + '/hubert_base.pt',
   RVC_ROOT / 'assets/hubert/hubert_base.pt')
dl(BASE + '/rmvpe.pt',
   RVC_ROOT / 'assets/rmvpe/rmvpe.pt')
dl(BASE + '/pretrained_v2/f0G' + SAMPLE_RATE + '.pth',
   RVC_ROOT / ('assets/pretrained_v2/f0G' + SAMPLE_RATE + '.pth'))
dl(BASE + '/pretrained_v2/f0D' + SAMPLE_RATE + '.pth',
   RVC_ROOT / ('assets/pretrained_v2/f0D' + SAMPLE_RATE + '.pth'))

print()
print('OK')

=== Symlinks ===
  assets -> /content/drive/MyDrive/rvc_standalone/assets
  logs -> /content/drive/MyDrive/rvc_standalone/logs
  datasets -> /content/drive/MyDrive/rvc_standalone/datasets

=== Tải weights ===
  skip (da co): hubert_base.pt
  skip (da co): rmvpe.pt
  skip (da co): f0G40k.pth
  skip (da co): f0D40k.pth

OK


In [5]:
# ── Cell 4: 🎙️ Chuẩn bị audio ────────────────────────────────────────────────
import shutil, subprocess, sys
from pathlib import Path

RVC_ROOT    = Path('/content/compare_rvc')
DATASET_DIR = RVC_ROOT / 'datasets' / EXPERIMENT
DATASET_DIR.mkdir(parents=True, exist_ok=True)

# Định dạng audio được hỗ trợ (sẽ tự convert sang .wav nếu cần)
SUPPORTED_EXT = {'.wav', '.mp3', '.flac', '.ogg', '.m4a', '.aac', '.opus', '.wma', '.aiff'}

def to_wav(src: Path, dst_dir: Path) -> Path:
    """Convert bất kỳ file audio nào sang .wav 44100Hz stereo."""
    dst = dst_dir / (src.stem + '.wav')
    if src.suffix.lower() == '.wav':
        shutil.copy2(src, dst)
    else:
        # dùng ffmpeg trực tiếp — có sẵn trên Colab, không cần pydub
        result = subprocess.run(
            ['ffmpeg', '-y', '-i', str(src), '-ar', '44100', '-ac', '2', str(dst)],
            capture_output=True, text=True
        )
        if result.returncode != 0:
            raise RuntimeError('ffmpeg failed on ' + src.name + ':\n' + result.stderr[-500:])
    return dst

# ── Nguồn audio ──────────────────────────────────────────────────────────────
if AUDIO_SRC == 'upload':
    try:
        from google.colab import files
        print('Chon file audio de upload (wav/mp3/flac/ogg/m4a/...):')
        uploaded = files.upload()
        tmp = Path('/tmp/rvc_upload')
        tmp.mkdir(exist_ok=True)
        for fname, data in uploaded.items():
            p = tmp / fname
            p.write_bytes(data)
            ext = p.suffix.lower()
            if ext not in SUPPORTED_EXT:
                print('  [skip] dinh dang khong ho tro:', fname)
                continue
            out = to_wav(p, DATASET_DIR)
            print('  ok:', fname, '->', out.name)
    except Exception as e:
        if 'widget' in str(e).lower() or 'browser' in str(e).lower() or 'session' in str(e).lower():
            print('[!] files.upload() khong hoat dong trong moi truong nay (VSCode/terminal).')
        else:
            print('[!] Loi upload:', e)
        print()
        print('Hay dung AUDIO_SRC = "drive" thay the:')
        print('  1. Upload file audio len Google Drive vao thu muc:')
        print('    ', DRIVE_AUDIO)
        print('  2. Doi AUDIO_SRC = "drive" trong Cell 2')
        print('  3. Chay lai Cell 4')
        raise SystemExit(0)

elif AUDIO_SRC == 'drive':
    src = Path(DRIVE_AUDIO)
    if not src.exists():
        raise FileNotFoundError(
            'Khong tim thay: ' + str(src)
            + ' — tao thu muc nay tren Drive va dat file audio vao do'
        )
    count = 0
    skipped = 0
    for f in sorted(src.iterdir()):
        if f.suffix.lower() not in SUPPORTED_EXT:
            skipped += 1
            continue
        out = to_wav(f, DATASET_DIR)
        print('  ok:', f.name, '->', out.name)
        count += 1
    print('Copied/converted', count, 'file(s) from Drive', end='')
    if skipped:
        print('  (bo qua', skipped, 'file khong ho tro)', end='')
    print()

# ── Tổng kết ─────────────────────────────────────────────────────────────────
wavs = sorted(DATASET_DIR.glob('*.wav'))
print()
print('Dataset:', len(wavs), 'file(s) .wav san sang trong', str(DATASET_DIR))
for w in wavs[:8]:
    print(' ', w.name)
if len(wavs) > 8:
    print('  ...')
if len(wavs) == 0:
    print('[!] Khong co file nao — kiem tra lai thu muc nguon.')


  ok: truc.m4a -> truc.wav
Copied/converted 1 file(s) from Drive

Dataset: 1 file(s) .wav san sang trong /content/compare_rvc/datasets/my_voice
  truc.wav


In [6]:
# ── Cell 5: 🔧 Tiền xử lý audio ──────────────────────────────────────────────
import os, subprocess, sys
from pathlib import Path

RVC_ROOT = Path('/content/compare_rvc')
exp_dir  = str(RVC_ROOT / 'logs' / EXPERIMENT)
log_file = RVC_ROOT / 'logs' / EXPERIMENT / 'preprocess.log'
os.makedirs(exp_dir, exist_ok=True)
log_file.write_text('', encoding='utf-8')

sr_map = {'32k': 32000, '40k': 40000, '48k': 48000}

env = os.environ.copy()
env['PYTHONPATH'] = str(RVC_ROOT) + os.pathsep + env.get('PYTHONPATH', '')

cmd = [
    sys.executable,
    'infer/modules/train/preprocess.py',
    str(RVC_ROOT / 'datasets' / EXPERIMENT),
    str(sr_map[SAMPLE_RATE]),
    '4',
    exp_dir,
    'False',
    '3.7',
]
print('Preprocess ...')
result = subprocess.run(
    cmd,
    cwd=str(RVC_ROOT),
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
# In toàn bộ output của subprocess
if result.stdout:
    print(result.stdout)

if result.returncode not in (0, None):
    # Đọc thêm file log nếu có
    log_txt = log_file.read_text(encoding='utf-8', errors='replace') if log_file.exists() else ''
    if log_txt.strip():
        print('=== preprocess.log ===')
        print(log_txt[-3000:])
    raise RuntimeError('preprocess.py failed (exit ' + str(result.returncode) + ')')

print('Preprocess DONE')


Preprocess ...
/content/compare_rvc/datasets/my_voice 40000 4 /content/compare_rvc/logs/my_voice False 3.7
start preprocess
/content/compare_rvc/datasets/my_voice/truc.wav	-> Success
end preprocess

Preprocess DONE


In [9]:
# ── Cell 6: 🎵 Trích F0 + HuBERT features ────────────────────────────────────
import os, subprocess, sys
from pathlib import Path

RVC_ROOT = Path('/content/compare_rvc')
exp_log  = str(RVC_ROOT / 'logs' / EXPERIMENT)
log_file = RVC_ROOT / 'logs' / EXPERIMENT / 'extract_f0_feature.log'
log_file.write_text('', encoding='utf-8')

env = os.environ.copy()
env['PYTHONPATH'] = str(RVC_ROOT) + os.pathsep + env.get('PYTHONPATH', '')

def run_and_show(label, cmd):
    print('=' * 60)
    print('>>>', label)
    print('=' * 60)
    r = subprocess.run(
        cmd,
        cwd=str(RVC_ROOT),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    output = r.stdout or ''
    if output.strip():
        print(output)
    else:
        print('(khong co output tu subprocess)')
    if r.returncode not in (0, None):
        # Đọc log file
        if log_file.exists():
            log_txt = log_file.read_text(encoding='utf-8', errors='replace').strip()
            if log_txt:
                print('--- extract_f0_feature.log (3000 ky tu cuoi) ---')
                print(log_txt[-3000:])
        raise RuntimeError(label + ' FAILED (exit ' + str(r.returncode) + ')')
    print('>>>', label, 'OK')

# --- Bước 0: Test import fairseq (để xem patch có hoạt động không) ---
print('=== Kiem tra fairseq import ===')
test = subprocess.run(
    [
        sys.executable, '-c',
        'import sys; sys.path.insert(0, "/content/compare_rvc"); '
        'from infer.lib.fairseq_torch_load_compat import apply_fairseq_torch_load_compat; '
        'apply_fairseq_torch_load_compat(); '
        'import fairseq; '
        'print("fairseq OK:", fairseq.__version__)',
    ],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env,
)
print(test.stdout or '(no output)')
if test.returncode != 0:
    raise RuntimeError('fairseq import FAILED — xem loi phia tren')

# --- 6a. Extract F0 ---
run_and_show(
    'Extract F0 (' + F0_METHOD + ')',
    [sys.executable,
     'infer/modules/train/extract/extract_f0_print.py',
     exp_log, '4', F0_METHOD]
)

# --- 6b. Extract HuBERT features ---
import torch
device  = 'cuda' if torch.cuda.is_available() else 'cpu'
is_half = 'true' if device == 'cuda' else 'false'
print('device:', device, '  is_half:', is_half)

run_and_show(
    'Extract HuBERT features',
    [sys.executable,
     'infer/modules/train/extract_feature_print.py',
     device, '1', '0', '0', exp_log, VERSION, is_half]
)

print()
print('Extract DONE')


=== Kiem tra fairseq import ===
2026-06-26 05:44:12.702899: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/usr/local/lib/python3.12/dist-packages/fairseq/__init__.py", line 29, in <module>
    from fairseq.dataclass.initialize import hydra_init
  File "/usr/local/lib/python3.12/dist-packages/fairseq/dataclass/initialize.py", line 8, in <module>
    from hydra.core.config_store import ConfigStore
  File "/usr/local/lib/python3.12/dist-packages/hydra/__init__.py", line 5, in <module>
    from hydra import utils
  File "/usr/local/lib/python3.12/dist-packages/hydra/utils.py", line 10, in <module>
    from hydra._internal.utils import (
  File "/usr/local/lib/python3.

RuntimeError: fairseq import FAILED — xem loi phia tren

In [ ]:
# ── Cell 7: 🚀 Training ───────────────────────────────────────────────────────
import json, os, subprocess, sys
from pathlib import Path
from random import shuffle

RVC_ROOT = Path('/content/compare_rvc')
exp_dir  = str(RVC_ROOT / 'logs' / EXPERIMENT)
fea_dim  = 768 if VERSION == 'v2' else 256

env = os.environ.copy()
env['PYTHONPATH'] = str(RVC_ROOT) + os.pathsep + env.get('PYTHONPATH', '')

# --- Tạo filelist.txt ---
gt_dir   = exp_dir + '/0_gt_wavs'
feat_dir = exp_dir + '/3_feature' + str(fea_dim)
f0_dir   = exp_dir + '/2a_f0'
f0nsf    = exp_dir + '/2b-f0nsf'

names = (
    set(n.split('.')[0] for n in os.listdir(gt_dir))
    & set(n.split('.')[0] for n in os.listdir(feat_dir))
    & set(n.split('.')[0] for n in os.listdir(f0_dir))
    & set(n.split('.')[0] for n in os.listdir(f0nsf))
)
opt = []
for name in names:
    opt.append(
        gt_dir   + '/' + name + '.wav'
        + '|' + feat_dir + '/' + name + '.npy'
        + '|' + f0_dir   + '/' + name + '.wav.npy'
        + '|' + f0nsf    + '/' + name + '.wav.npy|0'
    )

mute = str(RVC_ROOT / 'logs/mute')
for _ in range(2):
    opt.append(
        mute + '/0_gt_wavs/mute' + SAMPLE_RATE + '.wav'
        + '|' + mute + '/3_feature' + str(fea_dim) + '/mute.npy'
        + '|' + mute + '/2a_f0/mute.wav.npy'
        + '|' + mute + '/2b-f0nsf/mute.wav.npy|0'
    )

shuffle(opt)
with open(exp_dir + '/filelist.txt', 'w', encoding='utf-8') as f:
    for line in opt:
        print(line, file=f)
print('Filelist:', len(opt), 'entries')

# --- Config JSON ---
# Theo logic goc: v2+40k van dung v1 config
if VERSION == 'v1' or SAMPLE_RATE == '40k':
    cfg_src = str(RVC_ROOT / ('configs/v1/' + SAMPLE_RATE + '.json'))
else:
    cfg_src = str(RVC_ROOT / ('configs/v2/' + SAMPLE_RATE + '.json'))
cfg_dst = exp_dir + '/config.json'
if not os.path.exists(cfg_dst):
    with open(cfg_src) as fin, open(cfg_dst, 'w') as fout:
        json.dump(json.load(fin), fout, indent=4)

# --- Chạy training ---
import torch
has_gpu = torch.cuda.is_available()
pg = str(RVC_ROOT / ('assets/pretrained_v2/f0G' + SAMPLE_RATE + '.pth'))
pd = str(RVC_ROOT / ('assets/pretrained_v2/f0D' + SAMPLE_RATE + '.pth'))

cmd = [
    sys.executable, 'infer/modules/train/train.py',
    '-e', EXPERIMENT,
    '-sr', SAMPLE_RATE,
    '-f0', '1',
    '-bs', str(BATCH_SIZE),
    '-te', str(TOTAL_EPOCHS),
    '-se', str(SAVE_EVERY),
    '-pg', pg, '-pd', pd,
    '-l', '1', '-c', '0', '-sw', '0',
    '-v', VERSION,
]
if has_gpu:
    cmd += ['-g', '0']

print('Training started ...')
result = subprocess.run(cmd, cwd=str(RVC_ROOT), env=env)
if result.returncode not in (0, None):
    raise RuntimeError('train.py failed (exit ' + str(result.returncode) + ')')
print('Training DONE')

In [ ]:
# ── Cell 8: 📦 FAISS index + Export model inference ──────────────────────────
import glob, os, sys
from pathlib import Path

RVC_ROOT = Path('/content/compare_rvc')
if str(RVC_ROOT) not in sys.path:
    sys.path.insert(0, str(RVC_ROOT))
os.chdir(str(RVC_ROOT))

from training_pipeline.params import TrainingParams
from training_pipeline.setup_env import bootstrap
from training_pipeline.steps import step_train_index

rvc_root, cfg = bootstrap()

params = TrainingParams(
    experiment_name   = EXPERIMENT,
    version           = VERSION,
    sample_rate_label = SAMPLE_RATE,
)

# --- FAISS index ---
print('Building FAISS index ...')
for info in step_train_index(rvc_root, cfg, params):
    print(info)
print('FAISS OK')

# --- Export model inference ---
print()
g_ckpts = sorted(glob.glob(str(RVC_ROOT / 'logs' / EXPERIMENT / 'G_*.pth')))
if g_ckpts:
    from infer.lib.train.process_ckpt import extract_small_model
    best = g_ckpts[-1]
    print('Checkpoint:', best)
    out = extract_small_model(best, EXPERIMENT, SAMPLE_RATE, 1, 'Trained', VERSION)
    print('Inference model:', out)
else:
    print('Khong tim thay G_*.pth — training chua hoan thanh?')

print()
print('Hoan thanh! Model luu tai:', str(RVC_ROOT / 'assets/weights/'))